## Notebook 06 - Player-match matrix
Núria Pascual Salas

**Content:** Adds two variables to the player centralities dataset for the player-level analysis: player position (goalkeeper, defender, midfielder, or forward), taken from the Starting XI events, and the team's match-level Switching Factor (SF), which captures playing style. Position is included because centrality values have different meanings across roles, while SF provides the team's tactical context. The output is the final player-match dataset used in the later analysis.

**Inputs:** player_centralities_per_match.csv (from notebook 4).

**Outputs:**
- outputs/csv/player_match_matrix.csv (final matrix for the player-level analysis)

In [1]:
from utils import *
import pandas as pd
import numpy as np
from collections import defaultdict
import os
import time

CSV_DIR = 'outputs/csv'
os.makedirs(CSV_DIR, exist_ok=True)

### 1. Load the player centralities CSV

In [2]:
df_master = pd.read_csv(f'{CSV_DIR}/player_centralities_per_match.csv', encoding='utf-8-sig')
print(f"Total rows: {len(df_master)}")
print(f"Columns: {list(df_master.columns)}")
print()
print(df_master.head().to_string(index=False))

Total rows: 11790
Columns: ['match_id', 'home_team', 'away_team', 'team_id', 'team_name', 'team_is_home', 'score', 'outcome', 'player_name', 'rank', 'pagerank', 'pagerank_pct', 'betweenness', 'minutes_played']

 match_id        home_team away_team  team_id        team_name  team_is_home score outcome               player_name  rank  pagerank  pagerank_pct  betweenness  minutes_played
  3894664 Deportivo Alavés   Granada      206 Deportivo Alavés          True   3-1     win Luis Jesús Rioja González     1  0.134968         13.50       0.3379              89
  3894664 Deportivo Alavés   Granada      206 Deportivo Alavés          True   3-1     win     Javier López Carballo     2  0.100009         10.00       0.2198              95
  3894664 Deportivo Alavés   Granada      206 Deportivo Alavés          True   3-1     win        Ander Guevara Lajo     3  0.094683          9.47       0.1703              68
  3894664 Deportivo Alavés   Granada      206 Deportivo Alavés          True   3-1   

### 2. Position extraction from Starting XI events

In [3]:
POSITION_CATEGORY = {
    # Goalkeeper
    1:  'GK',
    
    # Defenders
    2:  'DEF',  3:  'DEF',  4:  'DEF',  5:  'DEF',  6:  'DEF',
    7:  'DEF',  8:  'DEF',
    
    # Midfielders
    9:  'MID',  10: 'MID',  11: 'MID',  12: 'MID',  13: 'MID',
    14: 'MID',  15: 'MID',  16: 'MID',
    
    # Attacking midfielders / wide forwards
    17: 'MID',  18: 'MID',  19: 'MID',  20: 'MID',
    
    # Forwards
    21: 'FWD',  22: 'FWD',  23: 'FWD',  24: 'FWD',  25: 'FWD',
}


def get_player_positions(events, team_name):
    """
    Extract each player's position from Starting XI and Substitution events.
    Returns dict {player_name: position_category}.
    """
    positions = {}
    
    # Starting XI
    for e in events:
        if (e.get('type', {}).get('name') == 'Starting XI'
            and e.get('team', {}).get('name') == team_name):
            lineup = e.get('tactics', {}).get('lineup', [])
            for player_info in lineup:
                player_name = player_info.get('player', {}).get('name')
                position_id = player_info.get('position', {}).get('id')
                if player_name and position_id:
                    positions[player_name] = POSITION_CATEGORY.get(position_id, 'UNK')
    
    # Substitutions
    for e in events:
        if (e.get('type', {}).get('name') == 'Substitution'
            and e.get('team', {}).get('name') == team_name):
            player_in = e.get('substitution', {}).get('replacement', {}).get('name')
            if player_in and player_in not in positions:
                # Fallback: use the position from any event where this player appears
                for sub_e in events:
                    if (sub_e.get('player', {}).get('name') == player_in
                        and 'position' in sub_e
                        and 'id' in sub_e.get('position', {})):
                        pos_id = sub_e['position']['id']
                        positions[player_in] = POSITION_CATEGORY.get(pos_id, 'UNK')
                        break
                if player_in not in positions:
                    positions[player_in] = 'UNK'
    
    return positions

### 3. Extract positions and SF for every match-team

In [4]:
positions_lookup = {}
sf_lookup        = {}

start_time = time.time()
n_processed = 0

for m_id, events in stream_matches_from_zip(zip_path, folder_laliga, "_events.json"):
    teams_in_match = list({(e['team']['id'], e['team']['name'])
                           for e in events if 'team' in e})
    
    for team_id, team_name in teams_in_match:
        if team_id not in all_teams:
            continue
        
        # Positions for this team in this match
        positions = get_player_positions(events, team_name)
        positions_lookup[(m_id, team_id)] = positions
        
        # SF for this team in this match
        sf_value = calculate_match_sf(events, team_id)
        sf_lookup[(m_id, team_id)] = sf_value
    
    n_processed += 1
    if n_processed % 100 == 0:
        print(f"  Processed {n_processed} matches | elapsed {time.time()-start_time:.1f}s")

print(f"\nDone. {n_processed} matches processed in {time.time()-start_time:.1f}s")

  Processed 100 matches | elapsed 11.9s
  Processed 200 matches | elapsed 24.8s
  Processed 300 matches | elapsed 38.3s

Done. 380 matches processed in 49.6s


### 4. Add the new columns to the DataFrame

In [5]:
def lookup_position(row):
    key = (str(row['match_id']), row['team_id'])
    positions = positions_lookup.get(key, {})
    return positions.get(row['player_name'], 'UNK')


def lookup_sf(row):
    key = (str(row['match_id']), row['team_id'])
    return sf_lookup.get(key, np.nan)


df_master['position']  = df_master.apply(lookup_position, axis=1)
df_master['team_sf']   = df_master.apply(lookup_sf, axis=1)

# Also add a binary target for logistic regression
df_master['won']       = (df_master['outcome'] == 'win').astype(int)

print(df_master.head().to_string(index=False))
print(f"\nFinal columns: {list(df_master.columns)}")

 match_id        home_team away_team  team_id        team_name  team_is_home score outcome               player_name  rank  pagerank  pagerank_pct  betweenness  minutes_played position  team_sf  won
  3894664 Deportivo Alavés   Granada      206 Deportivo Alavés          True   3-1     win Luis Jesús Rioja González     1  0.134968         13.50       0.3379              89      MID 0.247409    1
  3894664 Deportivo Alavés   Granada      206 Deportivo Alavés          True   3-1     win     Javier López Carballo     2  0.100009         10.00       0.2198              95      DEF 0.247409    1
  3894664 Deportivo Alavés   Granada      206 Deportivo Alavés          True   3-1     win        Ander Guevara Lajo     3  0.094683          9.47       0.1703              68      MID 0.247409    1
  3894664 Deportivo Alavés   Granada      206 Deportivo Alavés          True   3-1     win Andoni Gorosabel Espinosa     4  0.094051          9.41       0.3104              95      DEF 0.247409    1
  389

### 5. Validation

In [6]:
# Position distribution
print("Position distribution:")
print(df_master['position'].value_counts())
print()

# Percentage of UNK positions
pct_unk = (df_master['position'] == 'UNK').sum() / len(df_master) * 100
print(f"Rows with UNK position: {pct_unk:.1f}%")
print()

# SF distribution
print("Team SF statistics:")
print(df_master['team_sf'].describe())
print(f"\nRows with team_sf = NaN: {df_master['team_sf'].isna().sum()}")

Position distribution:
position
MID    4766
DEF    3928
FWD    2320
GK      773
UNK       3
Name: count, dtype: int64

Rows with UNK position: 0.0%

Team SF statistics:
count    11790.000000
mean         0.189605
std          0.057037
min          0.068102
25%          0.147222
50%          0.180493
75%          0.229885
max          0.372449
Name: team_sf, dtype: float64

Rows with team_sf = NaN: 0


In [7]:
# Cross-tabulation: position × outcome
print("Cross-tab position × outcome:")
print(pd.crosstab(df_master['position'], df_master['outcome']))
print()

# Mean PageRank by position
print("Mean PageRank by position:")
print(df_master.groupby('position')['pagerank_pct'].agg(['mean', 'std', 'count']).round(2))

Cross-tab position × outcome:
outcome   draw  loss   win
position                  
DEF       1064  1428  1436
FWD        653   818   849
GK         216   282   275
MID       1358  1745  1663
UNK          1     1     1

Mean PageRank by position:
          mean   std  count
position                   
DEF       7.39  3.00   3928
FWD       5.04  2.75   2320
GK        4.09  1.44    773
MID       6.74  3.47   4766
UNK       2.27  0.60      3


### 6. Save the final player-match matrix

In [8]:
output_path = f'{CSV_DIR}/player_match_matrix.csv'
df_master.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"Saved player-match matrix to: {output_path}")
print(f"File size: {os.path.getsize(output_path) / 1024:.1f} KB")
print(f"Total rows: {len(df_master)}")

Saved player-match matrix to: outputs/csv/player_match_matrix.csv
File size: 1642.9 KB
Total rows: 11790
